In [1]:
import gymnasium as gym
import numpy as np
import cv2
import tensorflow as tf
from keras.layers import Dense , Input
from keras import Model

env = gym.make("CartPole-v1")

In [2]:

# ==========================================
# Q-Network Definition
# ==========================================
net_input = Input(shape=(4,))
x = Dense(64 , activation='relu')(net_input)
x = Dense(32 , activation='relu')(x)
net_output = Dense(2 , activation='linear')(x)
q_net = Model(inputs=net_input , outputs=net_output)


In [6]:

# ==========================================
# Parameters
# ==========================================
GAMMA = 0.99
ALPHA = 0.001
EPSILON = 1.0
EPSILON_DECAY = 1.001
NUM_EPISODES = 1000


In [4]:

# ==========================================
# Policy Function
# ==========================================
def policy(state, explore=0.0):
    # Epsilon-Greedy policy
    if tf.random.uniform(shape=(), maxval=1.0) <= explore:
        return np.random.randint(0, 2)
    
    # State already (1, 4) form mein aayega
    q_values = q_net(state)
    return int(tf.argmax(q_values[0]).numpy())


In [7]:
# ==========================================
# Custom Training Loop (Q-Learning with TF Gradients)
# ==========================================
for episode in range(NUM_EPISODES):
    total_reward = 0
    length_episode = 0
    
    # Reset env aur Shape ko (4,) se (1, 4) banayein
    state, info = env.reset()
    state = tf.expand_dims(tf.convert_to_tensor(state, dtype=tf.float32), 0)
    
    done = False
    while not done:
        # 1. Action loop ke andar choose karein (Q-learning style)
        action = policy(state, explore=EPSILON)
        
        # 2. Gym step (Unpacking 5 values)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        # Shape change to (1, 4) for NN
        next_state = tf.expand_dims(tf.convert_to_tensor(next_state, dtype=tf.float32), 0)
        
        # 3. Target Calculation (THE Q-LEARNING MAGIC 🪄)
        if done:
            target = float(reward)
        else:
            # Yahan humne next_action hata kar seedha np.max() lagaya hai
            target = reward + GAMMA * np.max(q_net(next_state)[0].numpy())
        
        # 4. Gradient Calculation
        with tf.GradientTape() as tape:
            current_q_values = q_net(state)
            current_q = current_q_values[0][action] 
            
        grads = tape.gradient(current_q, q_net.trainable_weights)
        
        # Delta = Target - Current
        delta = target - current_q.numpy()
        
        # 5. Weight Update Rule
        for j in range(len(grads)):
            q_net.trainable_weights[j].assign_add(ALPHA * delta * grads[j])
            
        # Agle step ke liye variables update (sirf state, kyunki action top pe choose ho raha hai)
        state = next_state
        
        # 6. Correct reward assignment
        total_reward += reward
        length_episode += 1
    
    # Print episode ke baad hoga
    print(f"Episode: {episode + 1}, Length: {length_episode}, Total Reward: {total_reward}, Epsilon: {EPSILON:.3f}")
    
    # (Assuming EPSILON_DECAY is defined in your parameters)
    EPSILON /= EPSILON_DECAY  

env.close()

Episode: 1, Length: 27, Total Reward: 27.0, Epsilon: 1.000
Episode: 2, Length: 21, Total Reward: 21.0, Epsilon: 0.999
Episode: 3, Length: 19, Total Reward: 19.0, Epsilon: 0.998
Episode: 4, Length: 17, Total Reward: 17.0, Epsilon: 0.997
Episode: 5, Length: 10, Total Reward: 10.0, Epsilon: 0.996
Episode: 6, Length: 11, Total Reward: 11.0, Epsilon: 0.995
Episode: 7, Length: 45, Total Reward: 45.0, Epsilon: 0.994
Episode: 8, Length: 11, Total Reward: 11.0, Epsilon: 0.993
Episode: 9, Length: 12, Total Reward: 12.0, Epsilon: 0.992
Episode: 10, Length: 36, Total Reward: 36.0, Epsilon: 0.991
Episode: 11, Length: 23, Total Reward: 23.0, Epsilon: 0.990
Episode: 12, Length: 31, Total Reward: 31.0, Epsilon: 0.989
Episode: 13, Length: 68, Total Reward: 68.0, Epsilon: 0.988
Episode: 14, Length: 13, Total Reward: 13.0, Epsilon: 0.987
Episode: 15, Length: 19, Total Reward: 19.0, Epsilon: 0.986
Episode: 16, Length: 20, Total Reward: 20.0, Epsilon: 0.985
Episode: 17, Length: 21, Total Reward: 21.0, Epsi

In [8]:
q_net.save("q_learning_cartpole_model.h5")
print("Model saved as 'q_learning_cartpole_model.h5'")

Model saved as 'q_learning_cartpole_model.h5'
